In [1]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 25
batch_size = 100
learning_rate = 0.001

transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(root='../../data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='../../data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [3]:
class SELayer(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        out = self.avg_pool(x).view(b, c)
        out = self.fc(out).view(b, c, 1, 1)

        return x * out.expand_as(x)


In [4]:
def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1, stride=stride,
                     bias=False)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride)
        self.batchnorm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.batchnorm2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.selayer = SELayer(out_channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.batchnorm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.batchnorm2(out)
        out = self.selayer(out)

        if self.downsample:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

In [5]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super().__init__()
        self.in_channels = 16
        self.conv = conv3x3(3, 16)
        self.batchnorm = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self.create_layer(block, 16, layers[0])
        self.layer2 = self.create_layer(block, 32, layers[1], 2)
        self.layer3 = self.create_layer(block, 64, layers[2], 2)
        self.avg_pool = nn.AvgPool2d(kernel_size=8)
        self.fc = nn.Linear(64, num_classes)

    def create_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if (stride != 1) or (self.in_channels != out_channels):
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels

        for i in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        self.in_channels = out_channels

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)
        out = self.batchnorm(out)
        out = self.relu(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)

        return out

In [6]:
model = ResNet(ResidualBlock, [2, 2, 2]).to(device).float()
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

decay = 0
model.train()
for epoch in range(epochs):
    if (epoch + 1) % 20 == 0:
        decay += 1
        optimizer.param_groups[0]['lr'] = learning_rate * (0.5 ** decay)

    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_func(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 100 == 0:
            print("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
                  .format(epoch + 1, epochs, i + 1, len(train_loader), loss.item()))

Epoch [1/25], Step [100/500] Loss: 1.5627
Epoch [1/25], Step [200/500] Loss: 1.5258
Epoch [1/25], Step [300/500] Loss: 1.2719
Epoch [1/25], Step [400/500] Loss: 1.2144
Epoch [1/25], Step [500/500] Loss: 1.2158
Epoch [2/25], Step [100/500] Loss: 1.0527
Epoch [2/25], Step [200/500] Loss: 1.0413
Epoch [2/25], Step [300/500] Loss: 1.0022
Epoch [2/25], Step [400/500] Loss: 0.8144
Epoch [2/25], Step [500/500] Loss: 0.8915
Epoch [3/25], Step [100/500] Loss: 1.0617
Epoch [3/25], Step [200/500] Loss: 0.8656
Epoch [3/25], Step [300/500] Loss: 0.7622
Epoch [3/25], Step [400/500] Loss: 0.7672
Epoch [3/25], Step [500/500] Loss: 0.9671
Epoch [4/25], Step [100/500] Loss: 0.7266
Epoch [4/25], Step [200/500] Loss: 0.7046
Epoch [4/25], Step [300/500] Loss: 0.6785
Epoch [4/25], Step [400/500] Loss: 0.8716
Epoch [4/25], Step [500/500] Loss: 0.8516
Epoch [5/25], Step [100/500] Loss: 0.7598
Epoch [5/25], Step [200/500] Loss: 0.6114
Epoch [5/25], Step [300/500] Loss: 0.6507
Epoch [5/25], Step [400/500] Loss:

In [7]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        (_, predicted) = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(outputs, labels)
        total += labels.size(0)

    print('Accuracy of the model on the test images: {} %'.format(100 * correct / total))

Accuracy of the model on the test images: 85.52999877929688 %
